# لماذا نبدأ دائماً بالـ Value Object (`core/candle.py`) قبل `api/client.py`؟

---

## 🏛️ المبدأ الهندسي: Domain-Driven Design (DDD)

هناك مبدأ هندسي أساسي يفسر هذا الترتيب يسمى **Domain-Driven Design** (التصميم الموجه بمجال العمل).

---

## 🍹 تشبيه مصنع العصير

تخيل أنك تبني مصنعاً لتعبئة العصير:

> **هل تشتري شاحنات النقل (`api/client.py`) قبل أن تحدد شكل وحجم الزجاجة التي ستملؤها وتبيعها للمستهلك (`core/candle.py`)؟**

**بالتأكيد لا** — لأن الشاحنة وظيفتها فقط نقل البضاعة، لكن **شكل العبوة وحجمها ومعايير جودتها هي جوهر عملك**.

---

## 🎯 تطبيق التشبيه على البرمجيات

| المكوّن | التشبيه | الدور |
|---|---|---|
| **Core (القلب)** | الزجاجة ومعايير الجودة | تطبيقك أنت، وقواعد عملك أنت. لا يعرف ولا يهتم إن كانت البيانات قادمة من TwelveData أو Yahoo Finance أو من ملف محلي |
| **API Client (وسيلة النقل)** | شاحنة النقل | مجرد شاحنة تجلب نصوصاً من الإنترنت |

### ⚠️ المشكلة لو بنينا الشاحنة أولاً

لو بنينا الشاحنة أولاً، عندما تعود بالبيانات **لن تجد قالباً تفرغها فيه**، وستضطر لتمرير قواميس عشوائية (`dict`) **تفسد باقي أجزاء البرنامج**.

---

## 🔬 تشريح ملف `core/candle.py` — ملاحظات فقط

### 1️⃣ تعريف الكلاس وخصائصه الأساسية

**الاستيرادات:**
- `from __future__ import annotations` — لدعم Forward References في Type Hints
- `dataclass` — لتبسيط تعريف الكلاس
- `datetime` — لنوع حقل الوقت
- `Decimal` — للدقة المالية
- `Any` — للـ Type Hint في القواميس
- `ValidationError` — من `core.exceptions`

**الحقول:**
- `pair: str` — زوج العملات
- `timestamp: datetime` — وقت الشمعة
- `open, high, low, close: Decimal` — أسعار OHLC
- `volume: int = 0` — حجم التداول (اختياري)

#### 🔍 شرح القرارات التصميمية

| العنصر | السبب |
|---|---|
| **`@dataclass`** | توفر علينا كتابة `__init__` و `__repr__` يدوياً |
| **`frozen=True`** | تجعل الكائن **غير قابل للتعديل** (Immutable) — حماية حاسمة في التطبيقات المالية حتى لا تُعدل الأسعار بالخطأ |
| **`Decimal` بدلاً من `float`** | الفارق في الخانة الرابعة (0.0001) = نقطة مكسب/خسارة (Pip). `float` يسبب أخطاء تقريب ثنائية، بينما `Decimal` يحسب بدقة البشر |

> 💡 **مثال على خطأ `float`**: `0.1 + 0.2 == 0.3` → `False!` (النتيجة 0.30000000000000004). في الفوركس، الفرق ده ممكن يمثل **آلاف الدولارات**.

---

### 2️⃣ فحص الصحة المالية (`__post_init__`)

**دالة `__post_init__`:**
- **تتحقق من**: `low > high` → خطأ
- **تتحقق من**: `open < 0` أو `close < 0` → خطأ
- **تتحقق من**: `volume < 0` → خطأ
- **ترمي**: `ValidationError` في كل حالة

#### 🔍 المسؤولية: **حارس البوابة**

- إذا أرسل الـ API بيانات تالفة أو معكوسة، الكائن **يرفض التكوّن من الأساس**.
- هذا **يمنع دخول أي بيانات مشوهة** لقاعدة البيانات.

#### 🎯 لماذا `__post_init__`؟

- `@dataclass` بيعمل `__init__` تلقائياً — لكنه مش بيسمح بمنطق تحقق جواه.
- الحل: `__post_init__` بتنفذ تلقائياً بعد `__init__` — المكان المثالي للـ validation.

---

### 3️⃣ مصنع التحويل من الـ API إلى كائن (`from_dict`)

**دالة `from_dict` (Class Method):**

**خطوات التحويل:**
1. **قراءة حقل `datetime`** من القاموس
2. **فحص نوعه**:
   - لو `str` وفيه مسافة → صيغة `%Y-%m-%d %H:%M:%S`
   - لو `str` وبدون مسافة → صيغة `%Y-%m-%d`
   - لو `datetime` → استخدامه مباشرة
   - خلاف ذلك → رفع `ValueError`
3. **إنشاء `cls(...)`** مع تحويل:
   - `pair`: من `symbol` أو `pair` أو `"EUR/USD"` → ثم `.upper()`
   - `open/high/low/close`: `Decimal(str(...))`
   - `volume`: `int(...)` مع default = 0
4. **صيد الأخطاء**: `KeyError` و `ValueError` → رفع `ValidationError` مع `from err`

#### 🔍 المسؤولية: **Factory Method**

يأخذ القاموس الخام من TwelveData ويحوله إلى كائن `Candle` جاهز ونظيف.

#### 🎯 الميزات الذكية

| الميزة | الفائدة |
|---|---|
| **دعم أكثر من صيغة تاريخ** | مرونة مع مصادر مختلفة |
| **`data.get("symbol", data.get("pair", ...))`** | دعم أكثر من مفتاح |
| **`.upper()`** | تطبيع اسم الزوج |
| **`Decimal(str(...))`** | التريكة الذهبية — تحويل آمن |
| **`raise ... from err`** | يحفظ الـ traceback الأصلي |

> ⚠️ **تريكة `Decimal(str(x))`**: `Decimal(0.1)` = تمثيل ثنائي غير دقيق. `Decimal(str(0.1))` = `Decimal("0.1")` بدقة كاملة. **دايماً حوّل لـ string الأول**.

---

### 4️⃣ التحويل العكسي لحفظ البيانات (`to_dict`)

**دالة `to_dict`:**
- **ترجع قاموساً** فيه كل الحقول
- **`timestamp`**: `self.timestamp.isoformat()` — صيغة ISO
- **`open/high/low/close`**: `str(self.open)` — لأن `Decimal` مش بيتحول لـ JSON مباشرة
- **`volume`**: `self.volume` مباشرة
- **`pair`**: `self.pair` مباشرة

#### 🔍 المسؤولية: **Serialization**

- تُستخدم لحفظ الشمعة في SQLite أو تحويلها لـ JSON.
- **`str(self.open)`** بدل `self.open`: لأن `Decimal` مش JSON-serializable.

---

## 🗺️ خريطة الطريق لما بعد هذا الملف

```mermaid
graph TD
    A[1. core/candle.py<br/>شكل القالب الداخلي] --> B[2. api/client.py<br/>شاحنة النقل العامة]
    B --> C[3. api/sources/twelvedata.py<br/>ربط الشاحنة بالمصدر]
    C --> D[4. storage/sqlite_storage.py<br/>حفظ البيانات]
    D --> E[5. services/candle_service.py<br/>منطق العمل]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#f9f,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333,stroke-width:2px
```

### 📋 تفصيل كل خطوة

| # | الملف | المسؤولية |
|---|---|---|
| **1** | `core/candle.py` **(نحن هنا الآن)** | حددنا شكل القالب الداخلي للبيانات |
| **2** | `api/client.py` | **شاحنة النقل العامة** — ترسل طلبات GET، تتحقق من Status Code، أخطاء الاتصال، Timeout |
| **3** | `api/sources/twelvedata.py` | ربط الشاحنة بـ TwelveData وتفريغ JSON في قوالب `Candle` |
| **4** | `storage/sqlite_storage.py` | حفظ الشموع في قاعدة بيانات دائمة |
| **5** | `services/candle_service.py` | منطق العمل — يجمع الـ API والـ Storage |

---

## 🎯 لماذا هذا الترتيب مهم؟ (ملخص فلسفي)

| المبدأ | التطبيق |
|---|---|
| **Core First** | ابدأ بالقلب — ما يهم تطبيقك فعلاً |
| **Dependency Inversion** | الطبقات الخارجية تعتمد على القلب، وليس العكس |
| **Single Source of Truth** | `Candle` هو المرجع الوحيد لشكل البيانات |
| **Fail Fast** | البيانات التالفة تُرفض عند البوابة، مش بعد ما تدخل |
| **Immutability** | البيانات المالية لا تُعدَّل بعد إنشائها |
| **Precision** | `Decimal` للأسعار، مش `float` |

---

## 💡 القاعدة الذهبية لحفظها

> **"ابنِ القالب قبل الشاحنة"**
> 
> حدد شكل بياناتك أولاً (Value Object)، ثم ابنِ الأدوات اللي تجيبها (API Client)، ثم الأدوات اللي تخزنها (Storage)، وأخيراً الأدوات اللي تستخدمها (Services).

الترتيب ده بيخليك:
- ✅ تكتب كود **قابل للاختبار** (تقدر تختبر `Candle` بدون API).
- ✅ **تغيّر مصدر البيانات** بدون ما تعدل باقي النظام.
- ✅ **تمنع الأخطاء** قبل ما تدخل النظام.
- ✅ **تفهم الدومين** قبل ما تفهم التقنيات.

In [1]:
import json

# Объединяем имя переменной (meta_data) и сохраняем JSON-строку
meta_data = """{
    "meta": {
        "symbol": "EUR/USD",
        "interval": "1day"
    }, 
    "values": [
        { 
            "datetime": "2026-10-10", 
            "open": "1.0817", 
            "high": "1.0865", 
            "low": "1.0803", 
            "close": "1.0842", 
            "volume": "125432" 
        }
    ]
}"""

# Превращаем строку в удобный словарь Python
data = json.loads(meta_data)

# Пример использования данных:
print(f"Валютная пара: {data['meta']['symbol']}")
print(f"Цена закрытия: {data['values'][0]['close']}")

Валютная пара: EUR/USD
Цена закрытия: 1.0842


In [4]:
from core.candle import Candle

# استخراج القاموس الخام للشمعة
raw_candle = data["values"][0]
print(raw_candle)

# إنشاء كائن Candle
candle = Candle.from_dict(raw_candle)

print(candle)

{'datetime': '2026-10-10', 'open': '1.0817', 'high': '1.0865', 'low': '1.0803', 'close': '1.0842', 'volume': '125432', 'symbol': 'EUR/USD'}
Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 10, 0, 0), open=Decimal('1.0817'), high=Decimal('1.0865'), low=Decimal('1.0803'), close=Decimal('1.0842'), volume=125432)


In [5]:
from decimal import Decimal
from datetime import datetime
from core.candle import Candle

candle = Candle(
    pair=data["meta"]["symbol"],
    timestamp=datetime.strptime(data["values"][0]["datetime"], "%Y-%m-%d"),
    open=Decimal(data["values"][0]["open"]),
    high=Decimal(data["values"][0]["high"]),
    low=Decimal(data["values"][0]["low"]),
    close=Decimal(data["values"][0]["close"]),
    volume=int(data["values"][0]["volume"]),
)

print(candle)

Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 10, 0, 0), open=Decimal('1.0817'), high=Decimal('1.0865'), low=Decimal('1.0803'), close=Decimal('1.0842'), volume=125432)


# 📡 `HttpClient` — ملخص معماري شامل

---

## 🎯 الفكرة في سطر واحد

> **`HttpClient` هو "شاحنة النقل" التي تجلب البيانات من الإنترنت، لكنها شاحنة مُدرَّبة: تعرف متى تتوقف (Timeout)، ومتى تعتذر (ConnectionError)، ومتى ترفض الحمولة (HTTPError)، وكل ذلك تُبلّغ عنه بلغة موحّدة (`NetworkError`).**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[services/candle_service.py<br/>منطق العمل] --> B[api/sources/twelvedata.py<br/>مصدر محدد]
    B --> C[api/client.py<br/>HttpClient]
    C --> D[requests.Session<br/>مكتبة خارجية]
    C --> E[core/logger.py<br/>تسجيل]
    C --> F[core/exceptions.py<br/>NetworkError]

    style C fill:#f9f,stroke:#333,stroke-width:3px
    style A fill:#bfb,stroke:#333,stroke-width:2px
    style B fill:#bbf,stroke:#333,stroke-width:2px
```

**الطبقة:** `api/client.py` هي **الطبقة الأساسية للشبكة** — كل مصدر بيانات (TwelveData، Yahoo، …) يمر عبرها. لا أحد يتصل بالإنترنت مباشرة إلا من خلالها.

---

## 🧩 المسؤوليات الخمس

| # | المسؤولية | كيف تُنفَّذ |
|---|---|---|
| **1** | **إرسال الطلبات** | `self._session.get(url, params, timeout)` |
| **2** | **تطبيق Timeout موحّد** | `timeout=self.timeout` في كل طلب |
| **3** | **رفض الردود الفاشلة** | `response.raise_for_status()` — يرفض 4xx/5xx |
| **4** | **تحويل الاستثناءات** | كل خطأ `requests` → `NetworkError` واحد |
| **5** | **التسجيل** | `logger.info` قبل، `logger.error` عند الفشل |

---

## 🔄 دورة حياة الطلب (Flowchart)

```mermaid
graph TD
    A[1. استدعاء client.get] --> B[2. logger.info: إرسال الطلب]
    B --> C[3. session.get مع timeout]
    C --> D{4. نجح الاتصال؟}
    
    D -- لا / Timeout --> E1[logger.error + raise NetworkError]
    D -- لا / ConnectionError --> E2[logger.error + raise NetworkError]
    D -- نعم --> F[5. raise_for_status]
    
    F --> G{6. الحالة 2xx؟}
    G -- لا / HTTPError --> E3[logger.error + raise NetworkError]
    G -- نعم --> H[7. response.json]
    
    H --> I{8. JSON صالح؟}
    I -- لا / RequestException --> E4[logger.error + raise NetworkError]
    I -- نعم --> J[9. return dict]
    
    style A fill:#bbf,stroke:#333,stroke-width:2px
    style J fill:#bfb,stroke:#333,stroke-width:3px
    style E1 fill:#f99,stroke:#333,stroke-width:2px
    style E2 fill:#f99,stroke:#333,stroke-width:2px
    style E3 fill:#f99,stroke:#333,stroke-width:2px
    style E4 fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🛡️ خريطة تحويل الاستثناءات

```mermaid
graph LR
    A1[requests.Timeout] --> B[NetworkError]
    A2[requests.ConnectionError] --> B
    A3[requests.HTTPError] --> B
    A4[requests.RequestException] --> B
    
    B --> C[المستخدم يصطاد نوعاً واحداً فقط]

    style B fill:#f9f,stroke:#333,stroke-width:3px
    style C fill:#bfb,stroke:#333,stroke-width:2px
```

> 💡 **الفائدة الكبرى**: المستدعي لا يحتاج أن يعرف إن كان الخطأ Timeout أو DNS أو 404 — يكفيه `except NetworkError`.

---

## 🧬 تشريح الكود — قرارات التصميم

| القرار | السبب |
|---|---|
| **`requests.Session()`** | **Connection Pooling** — إعادة استخدام TCP connection بدل فتح واحدة جديدة لكل طلب (أسرع 2-3x) |
| **`timeout` كـ Constructor param** | يمكن تعديله لكل `HttpClient` حسب الحاجة (API سريع ≠ API بطيء) |
| **`raise_for_status()`** | يرفض 4xx/5xx تلقائياً — لا حاجة لفحص `status_code` يدوياً |
| **`params: dict`** | تمرير Query Parameters بشكل نظيف بدل بناء URL يدوياً |
| **`if "response" in locals()`** | حماية — لو الخطأ حدث قبل `response = ...`، لا يوجد `response` للوصول لـ `status_code` |
| **`raise ... from err`** | يحفظ الـ traceback الأصلي — لا يفقده عند التحويل |
| **`logger.info` قبل + `logger.error` عند الفشل** | كل طلب له أثر في `logs/app.log` |

---

## ⚠️ نقاط ضعف محتملة (للمراجعة المستقبلية)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **لا Retry** | فشل مؤقت في الشبكة = فشل نهائي | إضافة `tenacity` أو Retry يدوي |
| **`data: dict` فقط** | لو الـ API يرجع `list` (مثل `/comments`) → `dict[str, Any]` غلط | توقيع `-> Any` أو `dict \| list` |
| **لا Rate Limiting** | API مثل TwelveData له حدود (8 req/min) | إضافة `time.sleep` أو token bucket |
| **لا Headers** | بعض APIs تحتاج `User-Agent` أو `Authorization` | إضافة `headers` كـ param |
| **`HTTPError` قد لا يكون فيه `response`** | لو رُفع يدوياً بدون response | فحص أدق |
| **`Session` غير مغلقة** | تسريب اتصالات | `close()` أو `__enter__`/`__exit__` |

---

## 🧪 اختبارات موصى بها

```mermaid
graph TD
    A[TestHttpClient] --> B[test_get_success]
    A --> C[test_timeout_raises]
    A --> D[test_connection_error]
    A --> E[test_404_raises]
    A --> F[test_params_passed]
    A --> G[test_logger_called]

    B --> B1[Mock: 200 + JSON]
    C --> C1[Mock: requests.Timeout]
    D --> D1[Mock: ConnectionError]
    E --> E1[Mock: 404 + raise_for_status]
    F --> F1[تحقق أن params وصلت]
    G --> G1[تحقق من استدعاء logger]

    style A fill:#bbf,stroke:#333,stroke-width:3px
```

كل الاختبارات تستخدم `unittest.mock.patch` — **لا اتصال شبكة حقيقي**.

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"`HttpClient` = غلاف (Wrapper) حول `requests` يوحّد 4 أشياء:**
> **1. Timeout — 2. رفض 4xx/5xx — 3. تحويل الأخطاء — 4. التسجيل"**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ بدون `HttpClient` | ✅ مع `HttpClient` |
|---|---|---|
| **Timeout** | `requests.get(url)` — قد يعلّق للأبد | `timeout=10` — يفشل بأدب |
| **404** | يرجع `response` فاضي، تكتشفها لاحقاً | `HTTPError` → `NetworkError` فوراً |
| **ConnectionError** | `requests.ConnectionError` خام | `NetworkError` موحّد |
| **تسجيل** | لا شيء | كل طلب في `logs/app.log` |
| **صيد الأخطاء** | `except (Timeout, ConnectionError, HTTPError, ...)` | `except NetworkError` |
| **اختبار** | صعب — يحتاج mock لكل دالة | سهل — `patch` واحد |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅ مكتمل] --> B[api/client.py<br/>✅ مكتمل]
    B --> C[api/sources/twelvedata.py<br/>🔜 التالي]
    C --> D[storage/sqlite_storage.py<br/>📋 قادم]
    D --> E[services/candle_service.py<br/>📋 قادم]
    E --> F[cli/commands.py<br/>📋 إضافة أوامر]

    style B fill:#bfb,stroke:#333,stroke-width:3px
    style C fill:#f9f,stroke:#333,stroke-width:2px
```

**الخطوة التالية**: `api/sources/twelvedata.py` — يستخدم `HttpClient` لجلب البيانات، ثم `Candle.from_dict` لتحويلها.

---

## 💡 مثال سريع للاستخدام

```python
from api.client import HttpClient
from core.candle import Candle

client = HttpClient(timeout=15.0)
raw = client.get("https://api.twelvedata.com/time_series", params={...})
candle = Candle.from_dict(raw["values"][0])
```

**ثلاثة أسطر = طلب شبكة آمن + كائن `Candle` جاهز.** 🎯

In [1]:
from api.client import HttpClient

client = HttpClient(timeout=5.0)
data = client.get('https://api.frankfurter.app/latest', params={'from': 'EUR', 'to': 'USD'})
print('HTTP Client Works Successfully!')
print('Fetched Data:', data)

[2026-10-12 00:08:55] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.frankfurter.app/latest


HTTP Client Works Successfully!
Fetched Data: {'amount': 1.0, 'base': 'EUR', 'date': '2026-10-09', 'rates': {'USD': 1.1206}}


# 📡 `TwelveDataSource` — ملخص معماري شامل

---

## 🎯 الفكرة في سطر واحد

> **`TwelveDataSource` هي "المترجم" بين لغة TwelveData (JSON خام) ولغة تطبيقك (`Candle`). تأخذ استجابة API، تتحقق من أخطائها الخاصة، تحوّل كل شمعة إلى `Candle`، وتتجاهل الشموع التالفة بدل ما تُسقط العملية كلها.**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[services/candle_service.py<br/>منطق العمل] --> B[api/sources/twelvedata.py<br/>TwelveDataSource]
    B --> C[api/client.py<br/>HttpClient]
    C --> D[TwelveData API<br/>الإنترنت]
    B --> E[core/candle.py<br/>Candle.from_dict]
    B --> F[core/config.py<br/>api_key]
    B --> G[core/exceptions.py<br/>APIError]

    style B fill:#f9f,stroke:#333,stroke-width:3px
    style E fill:#bfb,stroke:#333,stroke-width:2px
    style C fill:#bbf,stroke:#333,stroke-width:2px
```

**الطبقة:** `api/sources/` هي **طبقة المحوّلات (Adapters)** — كل مصدر بيانات له ملفه الخاص، وكلهم يعتمدون على `HttpClient` و `Candle`.

---

## 🧩 المسؤوليات الخمس

| # | المسؤولية | كيف تُنفَّذ |
|---|---|---|
| **1** | **بناء الطلب** | تجميع `params`: `symbol`, `interval`, `outputsize`, `apikey` |
| **2** | **جلب البيانات** | `self.http_client.get(BASE_URL, params)` |
| **3** | **فحص أخطاء API الداخلية** | فحص `raw_response.get("status") == "error"` |
| **4** | **التحويل إلى `Candle`** | `Candle.from_dict(item)` لكل عنصر |
| **5** | **التسامح مع التالف** | `except ValidationError` → تسجيل تحذير + تجاهل الشمعة |

---

## 🔄 دورة حياة الجلب (Flowchart)

```mermaid
graph TD
    A[1. استدعاء fetch_candles] --> B[2. بناء params]
    B --> C[3. http_client.get]
    C --> D{4. استجابة API؟}
    
    D -- NetworkError --> E1[يرتفع للمستدعي]
    D -- نجح --> F{5. status == error؟}
    
    F -- نعم --> E2[logger.error + raise APIError]
    F -- لا --> G[6. استخراج values]
    
    G --> H{7. values فاضية؟}
    H -- نعم --> E3[logger.warning + return list فاضية]
    H -- لا --> I[8. loop على كل شمعة]
    
    I --> J[9. item.symbol = pair]
    J --> K{10. Candle.from_dict نجح؟}
    
    K -- ValidationError --> L[logger.warning + تجاهل]
    K -- نعم --> M[أضف للقائمة]
    
    L --> N{11. فيه شموع تانية؟}
    M --> N
    N -- نعم --> I
    N -- لا --> O[12. logger.info + return candles]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style O fill:#bfb,stroke:#333,stroke-width:3px
    style E1 fill:#f99,stroke:#333,stroke-width:2px
    style E2 fill:#f99,stroke:#333,stroke-width:2px
    style E3 fill:#ff9,stroke:#333,stroke-width:2px
    style L fill:#ff9,stroke:#333,stroke-width:2px
```

---

## 🛡️ خريطة الأخطاء الثلاثة

```mermaid
graph LR
    A1[NetworkError<br/>من HttpClient] --> D[المستدعي]
    A2[APIError<br/>status=error من TwelveData] --> D
    A3[ValidationError<br/>داخل كل شمعة] --> B[logger.warning فقط]
    B --> C[تجاهل الشمعة واستمرار]
    
    style A1 fill:#f99,stroke:#333,stroke-width:2px
    style A2 fill:#f99,stroke:#333,stroke-width:2px
    style A3 fill:#ff9,stroke:#333,stroke-width:2px
    style C fill:#bfb,stroke:#333,stroke-width:2px
```

> 💡 **الفلسفة**: أخطاء البنية (شبكة/API) → **ترتفع** للمستدعي. أخطاء البيانات (شمعة واحدة) → **تُسجَّل وتُتجاهَل**. لا نُسقط عملية 100 شمعة بسبب شمعة واحدة تالفة.

---

## 🧬 تشريح الكود — قرارات التصميم

| القرار | السبب |
|---|---|
| **`http_client: HttpClient \| None = None`** | **Dependency Injection** — يسمح بتمرير Mock في الاختبارات |
| **`api_key: str \| None = None`** | الأولوية: param → config → `"demo"` |
| **`BASE_URL` كـ Class Constant** | لا تكرار للرابط — قابل للتعديل في مكان واحد |
| **فحص `status == "error"`** | TwelveData يرجع 200 OK حتى لو فيه خطأ! لازم فحص داخلي |
| **`raw_response.get("values", [])`** | default فاضي — يمنع `KeyError` |
| **`item["symbol"] = pair`** | الـ API يضع `symbol` في `meta` مش في كل شمعة |
| **`except ValidationError` داخل الحلقة** | **Fault Tolerance** — شمعة تالفة لا تُسقط الكل |
| **`logger.warning` مش `error`** | الشمعة التالفة ليست خطأ نظام، بل مشكلة بيانات |

---

## ⚠️ نقاط ضعف محتملة (للمراجعة)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **`values` قد تكون dict مش list** | لو `outputsize=1`، بعض APIs ترجع `dict` مباشرة | فحص النوع: `if isinstance(values, dict): values = [values]` |
| **لا Rate Limiting** | TwelveData مجاني: 8 req/min | إضافة `time.sleep(8)` بين الطلبات |
| **`api_key="demo"`** | 4 عملات فقط (EUR/USD, USD/JPY, ...) | التحقق من وجود المفتاح عند الإقلاع |
| **`interval="1day"`** | لا تحقق من القيم المسموحة | إضافة `Literal` type أو `Enum` |
| **لا `timezone` param** | TwelveData يرجع UTC افتراضياً — قد تحتاج توقيتاً محلياً | إضافة `timezone` في `params` |
| **`item["symbol"] = pair` يُعدّل الـ dict الأصلي** | قد يسبب آثاراً جانبية | `item = {**item, "symbol": pair}` |
| **لا Retry على فشل مؤقت** | فشل شبكة لحظي = فشل نهائي | `tenacity` أو retry يدوي |

---

## 🧪 خريطة الاختبارات الموصى بها

```mermaid
graph TD
    A[TestTwelveDataSource] --> B[test_fetch_success]
    A --> C[test_api_error_raises]
    A --> D[test_empty_values]
    A --> E[test_invalid_candle_skipped]
    A --> F[test_api_key_from_config]
    A --> G[test_default_api_key_demo]
    A --> H[test_network_error_propagates]

    B --> B1[Mock: 2 شموع صالحة]
    C --> C1[Mock: status=error]
    D --> D1[Mock: values=[]]
    E --> E1[Mock: 1 صالحة + 1 تالفة]
    F --> F1[Config يرجع مفتاح]
    G --> G1[Config فاضي → demo]
    H --> H1[HttpClient يرمي NetworkError]

    style A fill:#bbf,stroke:#333,stroke-width:3px
```

كل الاختبارات تستخدم `Mock HttpClient` — **لا اتصال شبكة حقيقي**.

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"`TwelveDataSource` = 3 خطوات:**
> **1. اطلب (HttpClient) — 2. افحص (status/values) — 3. حوّل (Candle.from_dict مع تجاهل التالف)"**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ بدون `TwelveDataSource` | ✅ مع `TwelveDataSource` |
|---|---|---|
| **جلب شموع** | كود مكرر في كل مكان | `data_source.fetch_candles()` |
| **تحويل لـ `Candle`** | يدوياً لكل شمعة | تلقائي داخل الحلقة |
| **شمعة تالفة** | تُسقط العملية كلها | تُسجل وتُتجاهَل |
| **API error** | تكتشفه لاحقاً | `APIError` فوراً |
| **api_key** | hardcoded | من Config |
| **اختبار** | يحتاج شبكة حقيقية | Mock واحد |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅ مكتمل] --> B[api/client.py<br/>✅ مكتمل]
    B --> C[api/sources/twelvedata.py<br/>✅ مكتمل]
    C --> D[storage/sqlite_storage.py<br/>🔜 التالي]
    D --> E[services/candle_service.py<br/>📋 قادم]
    E --> F[cli/commands.py<br/>📋 إضافة أوامر fetch/price]

    style C fill:#bfb,stroke:#333,stroke-width:3px
    style D fill:#f9f,stroke:#333,stroke-width:2px
```

**الخطوة التالية**: `storage/sqlite_storage.py` — حفظ الشموع في قاعدة بيانات دائمة.

---

## 💡 مثال سريع للاستخدام

```python
from api.sources.twelvedata import TwelveDataSource

# إنشاء المصدر (يقرأ api_key من config.json تلقائياً)
source = TwelveDataSource()

# جلب 5 شموع يومية لزوج EUR/USD
candles = source.fetch_candles("EUR/USD", interval="1day", limit=5)

for c in candles:
    print(f"{c.timestamp.date()} | O:{c.open} H:{c.high} L:{c.low} C:{c.close}")
```

**سطران = شموع جاهزة ككائنات `Candle`.** 🎯

---

## 🧭 الفلسفة المعمارية في جملة

> **"`TwelveDataSource` لا يعرف كيف يتصل بالشبكة (يفوّض لـ `HttpClient`)، ولا يعرف شكل الشمعة (يفوّض لـ `Candle`)، ولا يعرف كيف يقرأ الإعدادات (يفوّض لـ `Config`). مسؤوليته الوحيدة: الترجمة والتنسيق بين هذه المكونات."**

In [15]:
from api.sources.twelvedata import TwelveDataSource

source = TwelveDataSource()
candles = source.fetch_candles(pair='EUR/USD', limit=1)

if candles:
    c = candles[0]
    print('--- Candle Fetched Successfully ---')
    print('Pair:', c.pair)
    print('Timestamp:', c.timestamp)
    print('Close Price:', c.close, type(c.close))
    print('High Price:', c.high)
    print('Low Price:', c.low)
else:
    print('No candles returned.')

[2026-10-25 20:00:41] [INFO] [forex_tracker]: جلب الشموع لزوج EUR/USD من TwelveData (العدد: 1)...
[2026-10-25 20:00:41] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-10-25 20:00:43] [INFO] [forex_tracker]: تم بنجاح تحويل 1 شمعة إلى كائنات Candle.


--- Candle Fetched Successfully ---
Pair: EUR/USD
Timestamp: 2026-10-25 00:00:00
Close Price: 1.12028 <class 'decimal.Decimal'>
High Price: 1.12042
Low Price: 1.12018


# 💾 `SQLiteStorage` — ملخص معماري شامل

---

## 🎯 الفكرة في سطر واحد

> **`SQLiteStorage` هي "الخزنة الدائمة" للشموع. تنفّذ عقد `Storage` (connect / disconnect / is_connected) وتضيف عمليتين مخصصتين: `save_candle` و `get_candles`. تخزّن الأسعار كنصوص (`TEXT`) للحفاظ على دقة `Decimal`، وتستخدم `UNIQUE + ON CONFLICT REPLACE` لمنع التكرار.**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[services/candle_service.py<br/>منطق العمل] --> B[storage/sqlite_storage.py<br/>SQLiteStorage]
    B --> C[storage/base.py<br/>Storage ABC]
    B --> D[core/candle.py<br/>Candle]
    B --> E[core/config.py<br/>data_dir]
    B --> F[SQLite File<br/>data/candles.db]
    
    G[storage/memory_storage.py<br/>MemoryStorage] -.implements.-> C
    
    style B fill:#f9f,stroke:#333,stroke-width:3px
    style C fill:#bbf,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:2px
```

**الطبقة:** `storage/` هي **طبقة الاستمرارية (Persistence)** — كل التطبيقات (Memory / SQLite / Postgres لاحقاً) تلتزم بنفس العقد `Storage`.

---

## 🧩 المسؤوليات الست

| # | المسؤولية | كيف تُنفَّذ |
|---|---|---|
| **1** | **تنفيذ عقد `Storage`** | `connect`, `disconnect`, `is_connected` |
| **2** | **إنشاء الجدول والفهارس تلقائياً** | `_init_db()` مع `CREATE TABLE IF NOT EXISTS` |
| **3** | **حفظ شمعة** | `INSERT OR REPLACE` — يمنع التكرار |
| **4** | **استرجاع شموع** | `SELECT ... ORDER BY timestamp DESC LIMIT ?` |
| **5** | **الحفاظ على دقة `Decimal`** | تخزين `str(...)` في أعمدة `TEXT` |
| **6** | **تحويل موحد للأخطاء** | `sqlite3.Error` → `StorageError` |

---

## 🔄 دورة حياة الكائن (Lifecycle)

```mermaid
graph TD
    A[1. SQLiteStorage] --> B[2. Config: data_dir]
    B --> C[3. os.makedirs]
    C --> D[4. connect]
    D --> E{5. متصل؟}
    E -- نعم --> F[تجاهل]
    E -- لا --> G[6. sqlite3.connect + row_factory]
    G --> H[7. _init_db]
    H --> I[CREATE TABLE IF NOT EXISTS]
    I --> J[CREATE INDEX IF NOT EXISTS]
    J --> K[commit]
    K --> L[8. الكائن جاهز]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style L fill:#bfb,stroke:#333,stroke-width:3px
```

> ⚠️ **ملاحظة**: الـ `__init__` يستدعي `connect()` و `_init_db()` تلقائياً — أي `SQLiteStorage()` = اتصال + جدول جاهز. هذا **Eager Initialization**، مريح لكن يجب ألا ينسى المستخدم `disconnect()`.

---

## 🗄️ مخطط قاعدة البيانات

```mermaid
erDiagram
    CANDLES {
        INTEGER id PK "AUTOINCREMENT"
        TEXT pair "NOT NULL"
        TEXT timestamp "NOT NULL"
        TEXT open "NOT NULL"
        TEXT high "NOT NULL"
        TEXT low "NOT NULL"
        TEXT close "NOT NULL"
        INTEGER volume "DEFAULT 0"
    }
    
    INDEX idx_pair_timestamp {
        pair ASC
        timestamp DESC
    }
```

**قيود مهمة:**
- `UNIQUE(pair, timestamp) ON CONFLICT REPLACE` → إعادة إدخال نفس الشمعة **تستبدلها** بدل رفضها.
- `idx_pair_timestamp` → يسرّع `get_candles` كثيراً (الاستعلام الأكثر استخداماً).
- **كل الأسعار `TEXT`** → لماذا؟ للحفاظ على دقة `Decimal` كاملة (لو `REAL`، `Decimal("1.0817")` قد يصبح `1.0817000000000001`).

---

## 🔄 دورة حفظ شمعة (Flowchart)

```mermaid
graph TD
    A[1. save_candle] --> B{2. متصل؟}
    B -- لا --> C[connect]
    B -- نعم --> D[3. cursor]
    C --> D
    D --> E[4. INSERT OR REPLACE]
    E --> F[5. commit]
    F --> G[6. logger.info]
    
    E -. sqlite3.Error .-> H[logger.error + raise StorageError]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style G fill:#bfb,stroke:#333,stroke-width:3px
    style H fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🔄 دورة استرجاع شموع (Flowchart)

```mermaid
graph TD
    A[1. get_candles pair, limit] --> B{2. متصل؟}
    B -- لا --> C[connect]
    B -- نعم --> D[3. cursor]
    C --> D
    D --> E[4. SELECT ... ORDER BY DESC LIMIT]
    E --> F[5. loop على rows]
    F --> G[6. Candle مع fromisoformat + Decimal]
    G --> H[7. return list Candle]
    
    E -. sqlite3.Error .-> I[logger.error + raise StorageError]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333,stroke-width:3px
    style I fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🧬 تشريح الكود — قرارات التصميم

| القرار | السبب |
|---|---|
| **`row_factory = sqlite3.Row`** | يسمح بالوصول للأعمدة بالاسم (`row["pair"]`) بدل الفهرس (`row[0]`) — أوضح وأأمن |
| **`os.makedirs(default_dir, exist_ok=True)`** | ينشئ مجلد `data/` تلقائياً — لا يفشل لو موجود |
| **`db_path: str \| None = None`** | **Dependency Injection** — يسمح بتمرير `:memory:` للاختبارات |
| **الأسعار كـ `TEXT`** | **حفظ دقة `Decimal`** — `REAL` يفقد الدقة |
| **`timestamp.isoformat()`** | صيغة موحدة قابلة للاسترجاع بـ `fromisoformat` |
| **`UNIQUE(pair, timestamp) ON CONFLICT REPLACE`** | **Idempotency** — إعادة الجلب لا تكرر البيانات |
| **`CREATE INDEX` على `(pair, timestamp DESC)`** | يسرّع الاستعلام الأكثر شيوعاً |
| **`connect()` تلقائي في كل دالة** | **Lazy Reconnect** — لو انقطع الاتصال، الدالة التالية تعيده |
| **`raise StorageError ... from err`** | يحفظ traceback الأصلي للديباج |
| **`pair.upper()` في `get_candles`** | تطبيع المدخلات — لا حساسية لحالة الأحرف |

---

## ⚠️ نقاط ضعف محتملة (للمراجعة)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **لا `close()` تلقائي** | تسريب اتصال لو نسيت `disconnect()` | `__enter__` / `__exit__` أو `atexit` |
| **`commit()` بعد كل شمعة** | بطيء عند حفظ 1000 شمعة | `save_many()` مع `executemany` + commit واحد |
| **لا `PRAGMA journal_mode=WAL`** | قراءات متزامنة قد تُحجب | `PRAGMA journal_mode=WAL` عند الاتصال |
| **لا `PRAGMA foreign_keys=ON`** | لا يوجد حالياً FK — مشكلة لاحقاً | `PRAGMA foreign_keys=ON` |
| **`ON CONFLICT REPLACE` = حذف + إدراج** | يفقد `id` الأصلي | `ON CONFLICT DO UPDATE` (SQLite 3.24+) |
| **لا `check_same_thread=False`** | لا يعمل في multi-threading | `sqlite3.connect(..., check_same_thread=False)` + `Lock` |
| **`sqlite3.Error` فقط** | `DataError`, `IntegrityError` كلها subclasses — مغطاة | ✅ لا مشكلة |
| **لا `CREATE TABLE` لمصادر أخرى** | مصمم للشموع فقط | مقبول — YAGNI |
| **لا `timezone` في `timestamp`** | `fromisoformat` يفقد TZ لو موجود | تخزين TZ في `isoformat` — ✅ موجودة أصلاً |
| **`save_candle` واحد بواحد** | بطيء للمجموعات | إضافة `save_many(candles)` |

---

## 🧪 خريطة الاختبارات الموصى بها

```mermaid
graph TD
    A[TestSQLiteStorage] --> B[test_connect_disconnect]
    A --> C[test_save_and_get]
    A --> D[test_duplicate_replaces]
    A --> E[test_decimal_precision]
    A --> F[test_order_desc]
    A --> G[test_limit]
    A --> H[test_unknown_pair_returns_empty]
    A --> I[test_storage_error_on_bad_db]
    A --> J[test_in_memory_db]

    B --> B1[is_connected True/False]
    C --> C1[save ثم get]
    D --> D1[save مرتين → صف واحد]
    E --> E1[Decimal 1.0817 يبقى 1.0817]
    F --> F1[الأحدث أولاً]
    G --> G1[limit=2 يرجع 2]
    H --> H1[list فاضية]
    I --> I1[مسار غير صالح → StorageError]
    J --> J1[db_path=:memory:]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style J fill:#bfb,stroke:#333,stroke-width:2px
```

> 💡 **أهم اختبار**: استخدم `db_path=":memory:"` — قاعدة بيانات في الذاكرة، سريعة ومعزولة تماماً.

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"`SQLiteStorage` = 3 أفكار:**
> **1. عقد `Storage` + جدول تلقائي — 2. أسعار `TEXT` لدقة `Decimal` — 3. `UNIQUE + REPLACE` لمنع التكرار"**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ بدون `SQLiteStorage` | ✅ مع `SQLiteStorage` |
|---|---|---|
| **حفظ شمعة** | كتابة SQL يدوياً | `storage.save_candle(candle)` |
| **استرجاع** | بناء استعلام + تحويل يدوي | `storage.get_candles("EUR/USD", 10)` |
| **دقة Decimal** | `REAL` يفقد الدقة | `TEXT` — دقة كاملة |
| **تكرار** | صفوف مكررة | `UNIQUE + REPLACE` |
| **أخطاء** | `sqlite3.Error` خام | `StorageError` موحّد |
| **اختبار** | ملف DB حقيقي | `:memory:` — معزول وسريع |
| **فهرس** | استعلامات بطيئة | `idx_pair_timestamp` |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅ مكتمل] --> B[api/client.py<br/>✅ مكتمل]
    B --> C[api/sources/twelvedata.py<br/>✅ مكتمل]
    C --> D[storage/sqlite_storage.py<br/>✅ مكتمل]
    D --> E[services/candle_service.py<br/>🔜 التالي]
    E --> F[cli/commands.py<br/>📋 إضافة fetch/price]

    style D fill:#bfb,stroke:#333,stroke-width:3px
    style E fill:#f9f,stroke:#333,stroke-width:2px
```

**الخطوة التالية**: `services/candle_service.py` — **منطق العمل** الذي يجمع `TwelveDataSource` + `SQLiteStorage`.

---

## 💡 مثال سريع للاستخدام

```python
from storage.sqlite_storage import SQLiteStorage
from api.sources.twelvedata import TwelveDataSource

# 1. تخزين في الذاكرة (للاختبار)
storage = SQLiteStorage(db_path=":memory:")

# 2. جلب شموع
source = TwelveDataSource()
candles = source.fetch_candles("EUR/USD", limit=5)

# 3. حفظها
for c in candles:
    storage.save_candle(c)

# 4. استرجاعها
saved = storage.get_candles("EUR/USD", limit=10)
print('--- Saved Candles ---')
print(saved)
for c in saved:
    print(f"{c.timestamp.date()} | C:{c.close}")

# 5. تنظيف
storage.disconnect()
```

**5 أسطر = حفظ دائم + استرجاع دقيق.** 🎯

---

## 🧭 الفلسفة المعمارية في جملة

> **"`SQLiteStorage` لا يعرف من أين أتت الشموع (TwelveData؟ ملف؟ مستخدم؟)، ولا يعرف شكلها (يفوّض لـ `Candle`)، ولا يعرف قواعد العمل (يفوّض للـ Services). مسؤوليته الوحيدة: تحويل كائن `Candle` إلى صف في SQLite، وإعادته ككائن `Candle` عند الطلب — بدقة متناهية وبدون تكرار."**

In [3]:
from storage.sqlite_storage import SQLiteStorage
from api.sources.twelvedata import TwelveDataSource

# 1. تخزين في الذاكرة (للاختبار)
storage = SQLiteStorage(db_path=":memory:")

# 2. جلب شموع
source = TwelveDataSource()
candles = source.fetch_candles("EUR/USD", limit=5)

# 3. حفظها
for c in candles:
    storage.save_candle(c)

# 4. استرجاعها
saved = storage.get_candles("EUR/USD", limit=10)
print('--- Saved Candles ---')
print(saved)
for c in saved:
    print(f"{c.timestamp.date()} | C:{c.close}")

storage.disconnect()


[2026-10-25 20:21:03] [INFO] [forex_tracker]: تم فتح الاتصال بقاعدة SQLite: :memory:
[2026-10-25 20:21:03] [INFO] [forex_tracker]: جلب الشموع لزوج EUR/USD من TwelveData (العدد: 5)...
[2026-10-25 20:21:03] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم بنجاح تحويل 5 شمعة إلى كائنات Candle.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم حفظ شمعة EUR/USD في SQLite.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم حفظ شمعة EUR/USD في SQLite.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم حفظ شمعة EUR/USD في SQLite.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم حفظ شمعة EUR/USD في SQLite.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم حفظ شمعة EUR/USD في SQLite.
[2026-10-25 20:21:04] [INFO] [forex_tracker]: تم إغلاق اتصال SQLite بنجاح.


--- Saved Candles ---
[Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 25, 0, 0), open=Decimal('1.12025'), high=Decimal('1.12042'), low=Decimal('1.11988'), close=Decimal('1.12017'), volume=0), Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 24, 0, 0), open=Decimal('1.11945'), high=Decimal('1.12054'), low=Decimal('1.11912'), close=Decimal('1.12024'), volume=0), Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 23, 0, 0), open=Decimal('1.1217'), high=Decimal('1.12275'), low=Decimal('1.11856'), close=Decimal('1.11949'), volume=0), Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 22, 0, 0), open=Decimal('1.12336'), high=Decimal('1.12551'), low=Decimal('1.11893'), close=Decimal('1.12165'), volume=0), Candle(pair='EUR/USD', timestamp=datetime.datetime(2026, 10, 21, 0, 0), open=Decimal('1.12351'), high=Decimal('1.12412'), low=Decimal('1.12144'), close=Decimal('1.12332'), volume=0)]
2026-10-25 | C:1.12017
2026-10-24 | C:1.12024
2026-10-23 | C:1.

# 🧠 `ForexService` — ملخص معماري شامل

---

## 🎯 الفكرة في سطر واحد

> **`ForexService` هي "المايسترو" (The Orchestrator) — لا تجلب بيانات بنفسها، ولا تخزن بنفسها. مهمتها الوحيدة: تنسيق الرقصة بين `TwelveDataSource` (الذي يجلب) و `Storage` (الذي يحفظ/يسترجع)، ثم تقديم واجهة نظيفة (`get_current_price`, `fetch_and_store`, `get_history`) لأي مستخدم (CLI / API / Tests).**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[cli/commands.py<br/>أو أي مستخدم] --> B[services/forex_service.py<br/>ForexService]
    B --> C[api/sources/twelvedata.py<br/>TwelveDataSource]
    B --> D[storage/base.py<br/>Storage ABC]
    D --> E[storage/sqlite_storage.py<br/>SQLiteStorage]
    D --> F[storage/memory_storage.py<br/>MemoryStorage]
    C --> G[api/client.py<br/>HttpClient]
    
    style B fill:#f9f,stroke:#333,stroke-width:3px
    style D fill:#bbf,stroke:#333,stroke-width:2px
```

**الطبقة:** `services/` هي **طبقة التنسيق (Application Layer)** — تعرف **ماذا** نريد، لكن لا تعرف **كيف** (تفوّض ذلك للمصادر والتخزين).

---

## 🧩 المسؤوليات الثلاث

| # | المسؤولية | كيف تُنفَّذ |
|---|---|---|
| **1** | **تنسيق الجلب + الحفظ** | `fetch_and_store` يستدعي source ثم storage |
| **2** | **توفير واجهة عالية المستوى** | `get_current_price`, `get_history` |
| **3** | **تسجيل الأحداث** | `logger.info` قبل وبعد كل عملية |

**ما لا تفعله** (وهذا مهم):
- ❌ لا تتصل بالشبكة مباشرة → `TwelveDataSource`
- ❌ لا تكتب SQL مباشرة → `Storage`
- ❌ لا تعرف تفاصيل `HttpClient` أو `sqlite3`

---

## 🔄 مخطط العلاقات بين المكونات

```mermaid
graph LR
    A[ForexService] -->|1. fetch| B[TwelveDataSource]
    A -->|2. save| C[Storage]
    A -->|3. get| C
    B -->|يحوّل| D[Candle]
    C -->|يعيد| D
    
    style A fill:#f9f,stroke:#333,stroke-width:3px
    style D fill:#bfb,stroke:#333,stroke-width:2px
```

**`Candle` هي "العملة المشتركة"** — هي ما يمر بين كل الطبقات.

---

## 🔄 دورة `fetch_and_store` (Flowchart)

```mermaid
graph TD
    A[1. fetch_and_store pair, limit] --> B[2. logger.info]
    B --> C[3. source.fetch_candles]
    C --> D{4. نجح؟}
    D -- لا --> E[NetworkError / APIError يرتفع]
    D -- نعم --> F[5. loop على candle]
    F --> G[6. storage.save_candle]
    G --> H{7. فيه تانية؟}
    H -- نعم --> F
    H -- لا --> I[8. logger.info]
    I --> J[9. return candles]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style J fill:#bfb,stroke:#333,stroke-width:3px
    style E fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🔄 دورة `get_current_price` (Flowchart)

```mermaid
graph TD
    A[1. get_current_price pair] --> B[2. logger.info]
    B --> C[3. source.fetch_candles limit=1]
    C --> D{4. فيه شموع؟}
    D -- لا --> E[raise ValueError]
    D -- نعم --> F[5. return candles 0 .close]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:3px
    style E fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🔄 دورة `get_history` (Flowchart)

```mermaid
graph TD
    A[1. get_history pair, limit] --> B[2. logger.info]
    B --> C[3. storage.get_candles pair, limit]
    C --> D[4. return list Candle]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 🧬 تشريح الكود — قرارات التصميم

| القرار | السبب |
|---|---|
| **`source: TwelveDataSource \| None = None`** | **Dependency Injection** — يمكن تمرير Mock في الاختبارات |
| **`storage: Storage \| None = None`** | **Dependency Inversion** — يعتمد على ABC، لا على `SQLiteStorage` |
| **`source or TwelveDataSource()`** | Default ذكي — إن لم يُمرَّر، يُنشئ واحداً |
| **`storage or SQLiteStorage()`** | Default = SQLite (الإنتاج)، لكن يمكن استبداله بـ `MemoryStorage` |
| **`get_current_price` → `fetch_candles(limit=1)`** | أبسط طريقة للحصول على آخر سعر — لا API endpoint منفصل |
| **`raise ValueError` لو لا توجد شموع** | خطأ منطقي (لا بيانات) ≠ خطأ شبكة |
| **`for candle in candles: save_candle`** | بسيط وواضح — يمكن تحويله لـ `save_many` لاحقاً |
| **`return candles` بعد الحفظ** | يسمح للمستخدم بالاستخدام الفوري دون إعادة قراءة |
| **`logger.info` قبل/بعد** | تتبع كامل في `logs/app.log` |
| **`pair: str = "EUR/USD"`** | Default مناسب — لكن يُفضّل إجبار التمرير في الإنتاج |

---

## ⚠️ نقاط ضعف محتملة (للمراجعة)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **`fetch_and_store` يحفظ واحداً بواحد** | بطيء لـ 1000 شمعة | `storage.save_many(candles)` + `executemany` |
| **لا Transaction** | لو فشل حفظ شمعة 50، الأولى 49 محفوظة | `with transaction:` في `Storage` |
| **`ValueError` في `get_current_price`** | استثناء عام — يصعب التمييز | `raise APIError("لا توجد بيانات")` |
| **`pair` مش `.upper()`** | `"eur/usd"` قد يفشل | `pair = pair.upper()` في البداية |
| **لا Cache** | كل `get_current_price` = طلب شبكة | إضافة TTL Cache بسيط |
| **لا Retry** | فشل مؤقت = فشل نهائي | `tenacity` أو retry يدوي |
| **`SQLiteStorage()` في Default** | يفتح اتصال DB حتى لو المستخدم لا يحتاجه | Lazy init |
| **لا تنسيق بين الشموع** | `fetch_and_store` قد تحفظ شموعاً مكررة | `UNIQUE + REPLACE` يحلها — ✅ |
| **لا حد أقصى لـ `limit`** | `limit=10**6` = مشكلة | `if limit > 5000: raise ValueError` |
| **`get_history` لا يتحقق من الاتصال** | `SQLiteStorage` يتعامل تلقائياً — ✅ | لا مشكلة |

---

## 🧪 خريطة الاختبارات الموصى بها

```mermaid
graph TD
    A[TestForexService] --> B[test_get_current_price_success]
    A --> C[test_get_current_price_no_data]
    A --> D[test_fetch_and_store_saves]
    A --> E[test_fetch_and_store_returns]
    A --> F[test_get_history_delegates]
    A --> G[test_default_dependencies]
    A --> H[test_mock_source]
    A --> I[test_mock_storage]

    B --> B1[Mock source يرجع شمعة]
    C --> C1[Mock source يرجع list فاضية]
    D --> D1[Mock storage.save_candle يُستدعى]
    E --> E1[تحقق من return value]
    F --> F1[Mock storage.get_candles]
    G --> G1[تحقق أن source/storage موجودين]
    H --> H1[Inject MockSource]
    I --> I1[Inject MockStorage]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style H fill:#bfb,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:2px
```

> 💡 **أهم ميزة**: بفضل DI، كل اختبار يستخدم `Mock` — **لا شبكة، لا DB حقيقي**.

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"`ForexService` = 3 كلمات:**
> **1. ينسّق (source + storage) — 2. لا يعرف كيف (يعتمد على ABC) — 3. يعطي واجهة نظيفة (3 دوال فقط)"**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ بدون `ForexService` | ✅ مع `ForexService` |
|---|---|---|
| **جلب + حفظ** | `source.fetch` ثم `for c: storage.save` في CLI | `service.fetch_and_store()` |
| **السعر اللحظي** | 3 أسطر في CLI | `service.get_current_price()` |
| **التاريخ** | `storage.get_candles()` مباشرة | `service.get_history()` |
| **الاختبار** | Mock لكل من source و storage | Mock واحد للـ service |
| **إضافة Cache** | تعديل كل مكان | تعديل `ForexService` فقط |
| **تغيير Storage** | تعديل CLI | لا تعديل (DI) |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅ مكتمل] --> B[api/client.py<br/>✅ مكتمل]
    B --> C[api/sources/twelvedata.py<br/>✅ مكتمل]
    C --> D[storage/sqlite_storage.py<br/>✅ مكتمل]
    D --> E[services/forex_service.py<br/>✅ مكتمل]
    E --> F[cli/commands.py<br/>🔜 إضافة fetch/price/history]

    style E fill:#bfb,stroke:#333,stroke-width:3px
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

**الخطوة التالية**: `cli/commands.py` — إضافة أوامر `fetch`, `price`, `history` تستخدم `ForexService`.

---

## 💡 مثال سريع للاستخدام

```python
from services.forex_service import ForexService
from storage.memory_storage import MemoryStorage

# 1. الإنتاج: SQLiteStorage (default)
service = ForexService()

# 2. الاختبار: MemoryStorage
service = ForexService(storage=MemoryStorage())

# 3. السعر اللحظي
price = service.get_current_price("EUR/USD")
print(f"EUR/USD = {price}")

# 4. جلب + تخزين
candles = service.fetch_and_store("EUR/USD", limit=10)

# 5. التاريخ
history = service.get_history("EUR/USD", limit=5)
for c in history:
    print(f"{c.timestamp.date()} | C:{c.close}")
```

**3 أسطر = السعر، الجلب، والتاريخ.** 🎯

---

## 🧭 الفلسفة المعمارية في جملة

> **"`ForexService` لا تعرف كيف يُرسَل HTTP (تفوّض لـ `HttpClient`)، ولا كيف يُحفَظ صف (تفوّض لـ `SQLiteStorage`)، ولا كيف تُبنى شمعة (تفوّض لـ `Candle`). مسؤوليتها الوحيدة: تنسيق هذه المكونات في سيناريوهات عمل واضحة (سعر لحظي، جلب وتخزين، تاريخ) — وهذا هو جوهر الطبقة الخدمية في Clean Architecture."**

In [4]:
from services.forex_service import ForexService
from storage.memory_storage import MemoryStorage

# 1. الإنتاج: SQLiteStorage (default)
service = ForexService()

# 2. الاختبار: MemoryStorage
service = ForexService(storage=MemoryStorage())

# 3. السعر اللحظي
price = service.get_current_price("EUR/USD")
print(f"EUR/USD = {price}")

# 4. جلب + تخزين
candles = service.fetch_and_store("EUR/USD", limit=10)

# 5. التاريخ
history = service.get_history("EUR/USD", limit=5)
for c in history:
    print(f"{c.timestamp.date()} | C:{c.close}")

[2026-10-25 20:25:43] [INFO] [forex_tracker]: تم فتح الاتصال بقاعدة SQLite: ./data\candles.db
[2026-10-25 20:25:43] [INFO] [forex_tracker]: طلب السعر اللحظي لزوج EUR/USD عبر ForexService...
[2026-10-25 20:25:43] [INFO] [forex_tracker]: جلب الشموع لزوج EUR/USD من TwelveData (العدد: 1)...
[2026-10-25 20:25:43] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-10-25 20:25:45] [INFO] [forex_tracker]: تم بنجاح تحويل 1 شمعة إلى كائنات Candle.
[2026-10-25 20:25:45] [INFO] [forex_tracker]: بدء عملية جلب وتخزين 10 شمعة لزوج EUR/USD...
[2026-10-25 20:25:45] [INFO] [forex_tracker]: جلب الشموع لزوج EUR/USD من TwelveData (العدد: 10)...
[2026-10-25 20:25:45] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series


EUR/USD = 1.12016


[2026-10-25 20:25:45] [INFO] [forex_tracker]: تم بنجاح تحويل 10 شمعة إلى كائنات Candle.
[2026-10-25 20:25:45] [INFO] [forex_tracker]: تم بنجاح جلب وتخزين 10 شمعة لزوج EUR/USD.
[2026-10-25 20:25:45] [INFO] [forex_tracker]: قراءة آخر 5 شمعة لزوج EUR/USD من قاعدة البيانات...


2026-10-20 | C:1.12351
2026-10-19 | C:1.12113
2026-10-18 | C:1.1176
2026-10-17 | C:1.12389
2026-10-16 | C:1.1239


# 🖥️ الخطوة 6 من Sprint 1 — تفعيل أوامر الـ CLI

---

## 🎯 الفكرة في سطر واحد

> **هذه الخطوة تربط "العقل" (`ForexService`) بـ "الوجه" (`CLI`). المستخدم يكتب `forex price` فيرى السعر، `forex fetch` فيجلب ويحفظ، `forex list` فيرى جدولاً منسقاً — وكل ذلك عبر 3 دوال بسيطة في `commands.py` + 3 أوامر مسجلة في `parser.py`.**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[User: forex price --pair EUR/USD] --> B[cli/parser.py<br/>build_parser + main]
    B --> C[cli/commands.py<br/>cmd_price / cmd_fetch / cmd_list]
    C --> D[services/forex_service.py<br/>ForexService]
    D --> E[TwelveDataSource<br/>أو Storage]
    
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#f9f,stroke:#333,stroke-width:3px
    style D fill:#bfb,stroke:#333,stroke-width:2px
```

**الطبقة:** `cli/` هي **طبقة العرض (Presentation)** — تعرض النتائج للمستخدم، وتلتقط الأخطاء، لكن **لا تحتوي منطق عمل**.

---

## 🧩 المسؤوليات الموزعة

| الملف | المسؤولية | ما لا يفعله |
|---|---|---|
| **`cli/commands.py`** | تنفيذ منطق كل أمر (استدعاء Service + طباعة + معالجة أخطاء) | ❌ لا يعرف argparse |
| **`cli/parser.py`** | تعريف الأوامر والمعاملات + Dispatch | ❌ لا يعرف Service |
| **`services/forex_service.py`** | منطق العمل (جلب/حفظ/استرجاع) | ❌ لا يعرف CLI |

---

## 🔄 دورة تنفيذ الأمر (Flowchart)

```mermaid
graph TD
    A[1. User: forex fetch --limit 3] --> B[2. main → parse_args]
    B --> C[3. args.func = cmd_fetch]
    C --> D[4. cmd_fetch: getattr pair/limit]
    D --> E[5. ForexService]
    E --> F[6. service.fetch_and_store]
    F --> G{7. نجح؟}
    
    G -- ForexError --> H[logger.error + print stderr + exit 1]
    G -- نعم --> I[8. print candles]
    
    style A fill:#bbf,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:3px
    style H fill:#f99,stroke:#333,stroke-width:2px
```

---

## 🔄 خريطة الأوامر الخمسة

```mermaid
graph LR
    A[forex] --> B[version]
    A --> C[status]
    A --> D[price]
    A --> E[fetch]
    A --> F[list]
    
    D --> D1[--pair EUR/USD]
    E --> E1[--pair + --limit]
    F --> F1[--pair + --limit]

    style D fill:#bfb,stroke:#333,stroke-width:2px
    style E fill:#bfb,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:2px
```

---

## 🧬 تشريح `commands.py` — قرارات التصميم

| القرار | السبب |
|---|---|
| **`argparse.Namespace` كـ type hint** | وضوح تام لنوع `args` — يساعد IDE والـ type checker |
| **`getattr(args, "pair", "EUR/USD")`** | default آمن — لا `AttributeError` لو الأمر بدون `--pair` |
| **`.upper()` على `pair`** | تطبيع — `eur/usd` = `EUR/USD` |
| **`ForexService()` جوه كل دالة** | بسيط — لا shared state بين الأوامر |
| **`try/except ForexError`** | يلتقط **كل** أخطاء التطبيق (Network/API/Storage) في مكان واحد |
| **`print(..., file=sys.stderr)`** | الأخطاء تروح لـ stderr — لا تختلط بـ stdout |
| **`sys.exit(1)` بعد الخطأ** | Exit code = 1 → CI/CD يعرف أن الأمر فشل |
| **`logger.error` قبل `print`** | أثر دائم في `logs/app.log` |
| **جدول `:<20` formatting** | محاذاة احترافية — أعمدة مستقيمة |
| **`if not candles` في `cmd_list`** | رسالة ودية بدل جدول فاضي |
| **`str(c.open)` في الجدول** | `Decimal` لا يُنسّق مباشرة في f-string مع `<10` |

---

## 🧬 تشريح `parser.py` — قرارات التصميم

| القرار | السبب |
|---|---|
| **`subparsers.add_parser(...)`** | كل أمر فرعي له `--help` مستقل |
| **`p_price.add_argument("--pair", ...)`** | تمرير اختياري — الـ default موجود |
| **`p_fetch.add_argument("--limit", type=int, ...)`** | `type=int` → argparse يرفض غير الأرقام تلقائياً |
| **`set_defaults(func=...)`** | **Dispatch Pattern** — ربط أمر بدالة |
| **`dest="command"`** | يخزّن اسم الأمر في `args.command` (للتشخيص) |
| **`hasattr(args, "func")`** | فحص أمان — لو أمر غير معروف |
| **`if len(sys.argv) == 1`** | بدون أوامر → Help تلقائي |

---

## ⚠️ نقاط ضعف محتملة (للمراجعة)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **`ForexService()` في كل دالة** | يفتح اتصال SQLite جديد في كل أمر | مشاركة instance أو lazy singleton |
| **`try/except ForexError` فقط** | `ValueError` في `cmd_price` قد يفلت | `except ForexError` ثم `except Exception` |
| **لا `finally: service.storage.disconnect()`** | SQLite قد يبقى مفتوحاً | `try/finally` |
| **`print(f"\n💵 ...")`** | Emoji قد لا يظهر في بعض Terminals | `--no-emoji` flag |
| **`limit` بدون حد أقصى** | `forex fetch --limit 100000` = مشكلة | `if limit > 5000: raise` |
| **`pair` بدون تحقق** | `EURUSD` أو `eur/usd` قد يفشل | regex validation |
| **جدول `:<20` ثابت** | التواريخ الطويلة قد تكسر المحاذاة | `tabulate` أو `rich` library |
| **لا `--json` flag** | لا يمكن استخدام الأوامر في scripts | `--json` → `json.dumps(candle.to_dict())` |
| **`sys.exit(1)` داخل command** | يصعّب الاختبار (يوقف العملية) | `raise SystemExit` أو `return int` |
| **`cmd_status` لا يتحقق من الاتصال** | يقول "Ready" حتى لو DB معطلة | فحص `storage.is_connected()` |

---

## 🧪 خريطة الاختبارات الموصى بها

```mermaid
graph TD
    A[TestCLICommands] --> B[test_cmd_version_output]
    A --> C[test_cmd_price_success]
    A --> D[test_cmd_price_forex_error]
    A --> E[test_cmd_fetch_saves]
    A --> F[test_cmd_list_empty]
    A --> G[test_cmd_list_with_data]
    A --> H[test_cmd_pair_uppercase]
    A --> I[test_cmd_default_pair]
    A --> J[test_parser_registers_5_commands]

    B --> B1[capture stdout]
    C --> C1[Mock ForexService]
    D --> D1[Mock يرمي ForexError]
    E --> E1[تحقق من استدعاء fetch_and_store]
    F --> F1[رسالة No saved candles]
    G --> G1[تحقق من شكل الجدول]
    H --> H1[eur/usd → EUR/USD]
    I --> I1[بدون --pair → EUR/USD]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style D fill:#f99,stroke:#333,stroke-width:2px
```

> 💡 **أهم تريكة اختبار**: استخدم `unittest.mock.patch("cli.commands.ForexService")` لعزل الـ CLI عن الشبكة وDB.

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"`commands.py` = 3 عناصر:**
> **1. قراءة `args` — 2. استدعاء `Service` — 3. طباعة + معالجة `ForexError`"**
> 
> **"`parser.py` = 2 عناصر:**
> **1. تعريف أوامر ومعاملات — 2. `set_defaults(func=...)` للـ Dispatch"**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ قبل الخطوة 6 | ✅ بعد الخطوة 6 |
|---|---|---|
| **السعر اللحظي** | لا يوجد أمر | `forex price` |
| **جلب شمعة** | يدوياً في Python | `forex fetch --limit 3` |
| **عرض التاريخ** | `storage.get_candles()` يدوياً | `forex list` |
| **معالجة الأخطاء** | `Traceback` خام | `Error: ...` + exit 1 |
| **تسجيل** | لا شيء | `logs/app.log` |
| **تنسيق الجدول** | لا شيء | محاذاة احترافية |
| **قائمة المساعدة** | لا شيء | `forex --help` |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅] --> B[api/client.py<br/>✅]
    B --> C[api/sources/twelvedata.py<br/>✅]
    C --> D[storage/sqlite_storage.py<br/>✅]
    D --> E[services/forex_service.py<br/>✅]
    E --> F[cli/commands.py + parser.py<br/>✅]
    F --> G[tests/test_price_mvp.py<br/>🔜 التالي]

    style F fill:#bfb,stroke:#333,stroke-width:3px
    style G fill:#f9f,stroke:#333,stroke-width:2px
```

**الخطوة التالية**: `tests/test_price_mvp.py` — اختبارات الوحدة المؤتمتة، وإغلاق Sprint 1.

---

## 💡 مثال سريع للاستخدام

```bash
# 1. المساعدة
forex --help

# 2. السعر اللحظي
forex price
forex price --pair GBP/USD

# 3. جلب وحفظ
forex fetch --limit 3
forex fetch --pair USD/JPY --limit 5

# 4. عرض التاريخ
forex list
forex list --pair EUR/USD --limit 20
```

**المتوقع من `forex list`:**
```
--- Stored Candles for EUR/USD (Last 3) ---
Date / Time          | Open       | High       | Low        | Close      | Volume
----------------------------------------------------------------------------------
2026-10-10 00:00     | 1.0817     | 1.0865     | 1.0803     | 1.0842     | 125432
2026-10-09 00:00     | 1.0800     | 1.0850     | 1.0780     | 1.0817     | 118900
2026-10-08 00:00     | 1.0790     | 1.0830     | 1.0760     | 1.0800     | 110450
```

---

## 🧭 الفلسفة المعمارية في جملة

> **"`cli/` هي وجه التطبيق فقط: `parser.py` يعرف 'ماذا طلب المستخدم'، `commands.py` يعرف 'كيف يستدعي Service ويعرض النتيجة'. لا أحدهما يعرف كيف يُرسَل HTTP أو كيف يُحفَظ صف — وهذا هو **Separation of Concerns** في أنقى صوره."**


# 🧪 `tests/test_price_mvp.py` — ملخص معماري شامل

---

## 🎯 الفكرة في سطر واحد

> **هذا الملف هو "الشبكة الأمان" لـ Sprint 1. يختبر ثلاث طبقات (Candle / SQLiteStorage / ForexService) في 3 فئات اختبار و6 اختبارات، باستخدام `unittest` فقط + `MagicMock` + قاعدة بيانات في الذاكرة (`:memory:`). لا شبكة، لا ملفات، لا تعتمد على ترتيب الاختبارات.**

---

## 🏗️ الموقع المعماري

```mermaid
graph TD
    A[tests/test_price_mvp.py] --> B[TestCandleModel]
    A --> C[TestSQLiteStorage]
    A --> D[TestForexService]
    
    B -->|يختبر| E[core/candle.py]
    C -->|يختبر| F[storage/sqlite_storage.py]
    D -->|يختبر| G[services/forex_service.py]
    D -->|يستخدم| H[MagicMock]
    D -->|يستخدم| F
    
    style A fill:#bbf,stroke:#333,stroke-width:3px
    style E fill:#bfb,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:2px
    style G fill:#bfb,stroke:#333,stroke-width:2px
```

**الطبقة:** `tests/` هي **شبكة الأمان (Safety Net)** — تحمي كل طبقة من الانحدار (Regression).

---

## 🧩 خريطة التغطية (Coverage Map)

```mermaid
graph LR
    A[TestCandleModel<br/>3 tests] --> A1[valid creation]
    A --> A2[invalid low>high]
    A --> A3[from_dict + to_dict]
    
    B[TestSQLiteStorage<br/>1 test] --> B1[save + get]
    
    C[TestForexService<br/>2 tests] --> C1[get_current_price]
    C --> C2[fetch_and_store]

    style A fill:#bfb,stroke:#333,stroke-width:2px
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#f9f,stroke:#333,stroke-width:2px
```

---

## 🔄 دورة حياة كل اختبار (Lifecycle)

```mermaid
graph TD
    A[1. setUp] --> B[2. test method]
    B --> C{3. نجح؟}
    C -- نعم --> D[4. tearDown]
    C -- AssertionError --> E[4. tearDown]
    D --> F[5. OK]
    E --> G[5. FAILED]
    
    style A fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#bbf,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:3px
    style G fill:#f99,stroke:#333,stroke-width:2px
```

> 💡 **`tearDown` تنفذ دائماً** — حتى لو الاختبار فشل. لهذا `storage.disconnect()` مضمون.

---

## 🧬 تشريح الفئات الثلاث

### 1️⃣ `TestCandleModel` — اختبارات الدومين

| الاختبار | ما يتحقق منه | الأداة |
|---|---|---|
| `test_candle_creation_valid` | `Candle` تُنشأ بـ `pair` و `close` صحيحين | `assertEqual` |
| `test_candle_invalid_low_greater_than_high` | `low > high` → `ValidationError` | `assertRaises` |
| `test_candle_from_dict_and_to_dict` | `from_dict` → `Candle` → `to_dict` دورة كاملة | `assertEqual` + `assertIn` |

**قرارات التصميم:**

| القرار | السبب |
|---|---|
| **لا `setUp`** | `Candle` عديمة الحالة (frozen) — لا حاجة لتهيئة |
| **`assertRaises(ValidationError)`** | يختبر **الرفض** لا النجاح فقط |
| **`assertIn("2026-10-25", serialized["timestamp"])`** | يتحقق من الصيغة بدون الاعتماد على `T00:00:00` |
| **`datetime(2026, 10, 25, 12, 0)`** | تاريخ ثابت — لا يعتمد على `now()` |

---

### 2️⃣ `TestSQLiteStorage` — اختبارات الاستمرارية

| الاختبار | ما يتحقق منه | الأداة |
|---|---|---|
| `test_save_and_get_candles` | حفظ شمعة → استرجاعها بنفس القيم | `assertEqual` |

**قرارات التصميم:**

| القرار | السبب |
|---|---|
| **`db_path=":memory:"`** | **التريكة الذهبية** — قاعدة بيانات في الذاكرة: سريعة، معزولة، لا تترك ملفات |
| **`setUp` ينشئ Storage جديداً** | كل اختبار في بيئة نظيفة |
| **`tearDown: disconnect()`** | تحرير الاتصال — لا تسريب |
| **`assertEqual(len(retrieved), 1)`** | يتحقق من **عدد الصفوف** لا القيمة فقط |

---

### 3️⃣ `TestForexService` — اختبارات التنسيق

| الاختبار | ما يتحقق منه | الأداة |
|---|---|---|
| `test_get_current_price` | `get_current_price` يستدعي `fetch_candles(limit=1)` ويرجع `close` | `MagicMock` + `assert_called_once_with` |
| `test_fetch_and_store_orchestration` | `fetch_and_store` يجلب **و** يحفظ فعلياً | `MagicMock` + `SQLiteStorage` حقيقي |

**قرارات التصميم:**

| القرار | السبب |
|---|---|
| **`MagicMock()` للـ source** | عزل الشبكة — لا اتصال حقيقي |
| **`SQLiteStorage(":memory:")` للـ storage** | اختبار حقيقي للحفظ (لا mock) |
| **`assert_called_once_with(pair=..., limit=1)`** | يتحقق من **المعاملات** لا الاستدعاء فقط |
| **`history = self.storage.get_candles(...)`** | يتحقق من **الأثر الجانبي** — الشمعة وصلت فعلاً للتخزين |
| **`fetch_candles.return_value = [fake_candle]`** | يحدد ما يعيده الـ Mock |

> 💡 **فلسفة `test_fetch_and_store_orchestration`**: الـ `source` مُموَّه (لا شبكة)، لكن الـ `storage` **حقيقي** — هذا اختبار **تكامل جزئي** (Partial Integration Test) يضمن أن التنسيق يعمل فعلاً.

---

## 🛡️ خريطة أنواع `assert` المستخدمة

```mermaid
graph LR
    A[assertEqual] --> A1[قيمة == قيمة]
    B[assertRaises] --> B1[استثناء متوقع]
    C[assertIn] --> C1[عنصر داخل]
    D[assert_called_once_with] --> D1[استدعاء Mock مرة واحدة بمعاملات محددة]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style B fill:#f99,stroke:#333,stroke-width:2px
    style C fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#f9f,stroke:#333,stroke-width:2px
```

---

## ⚠️ نقاط ضعف محتملة (للمراجعة)

| النقطة | التأثير | الحل المقترح |
|---|---|---|
| **لا اختبار `to_dict` كامل** | الحقول الأخرى (`open`, `high`, …) غير مختبرة | `assertEqual(serialized, expected_dict)` |
| **لا اختبار `volume < 0`** | قاعدة تحقق مفقودة | إضافة `test_candle_negative_volume` |
| **لا اختبار `open/close < 0`** | قاعدة تحقق مفقودة | إضافة `test_candle_negative_price` |
| **لا اختبار `from_dict` بحقول ناقصة** | `KeyError` → `ValidationError` غير مختبر | إضافة `test_from_dict_missing_field` |
| **لا اختبار `get_candles` مع `limit`** | ترتيب DESC غير مختبر | إضافة `test_get_candles_order_desc` |
| **لا اختبار `UNIQUE + REPLACE`** | حفظ نفس الشمعة مرتين | إضافة `test_save_duplicate_replaces` |
| **لا اختبار `pair.upper()`** | `"eur/usd"` غير مختبر | إضافة `test_lowercase_pair` |
| **`test_get_current_price` لا يختبر `ValueError`** | حالة `candles=[]` غير مختبرة | إضافة `test_get_current_price_empty` |
| **لا اختبار `cmd_*`** | CLI غير مختبر | إضافة `TestCLICommands` |
| **`MagicMock()` بدون `spec`** | يقبل أي استدعاء — لا يكتشف أخطاء API | `MagicMock(spec=TwelveDataSource)` |
| **`datetime.now()` في `test_fetch_and_store`** | غير حتمي — لكن مقبول لأنه لا يُقارن | استخدام تاريخ ثابت |
| **`disconnect()` في `tearDown` فقط** | لو `setUp` فشل، لا `tearDown` | مقبول — `setUp` بسيط |

---

## 🧪 خريطة الاختبارات الموسعة الموصى بها

```mermaid
graph TD
    A[TestCandleModel] --> A1[valid]
    A --> A2[low > high]
    A --> A3[from_dict + to_dict]
    A --> A4[negative volume]
    A --> A5[negative price]
    A --> A6[missing field in from_dict]
    A --> A7[frozen immutable]
    
    B[TestSQLiteStorage] --> B1[save + get]
    B --> B2[duplicate replaces]
    B --> B3[order desc]
    B --> B4[limit works]
    B --> B5[unknown pair empty]
    B --> B6[disconnect then save reconnects]
    
    C[TestForexService] --> C1[get_current_price]
    C --> C2[get_current_price empty → ValueError]
    C --> C3[fetch_and_store orchestration]
    C --> C4[get_history delegates]
    C --> C5[default dependencies]
    
    D[TestCLICommands] --> D1[cmd_price success]
    D --> D2[cmd_price ForexError]
    D --> D3[cmd_fetch saves]
    D --> D4[cmd_list empty]
    D --> D5[cmd_list with data]

    style A fill:#bfb,stroke:#333,stroke-width:2px
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#f9f,stroke:#333,stroke-width:2px
    style D fill:#ff9,stroke:#333,stroke-width:2px
```

---

## 🎁 ملخص "لو نسيت كل حاجة، افتكر دي"

> ### 🧠 القاعدة الذهبية
> **"3 فئات = 3 طبقات:**
> **1. `Candle` = منطق الدومين — 2. `SQLiteStorage` = استمرارية (`:memory:`) — 3. `ForexService` = تنسيق (Mock للشبكة + Storage حقيقي)"**
> 
> **"`assertRaises` للرفض، `assertEqual` للقيمة، `assert_called_once_with` للتعاون."**

---

## 📊 جدول المقارنة: قبل وبعد

| السيناريو | ❌ بدون اختبارات | ✅ مع الاختبارات |
|---|---|---|
| **تعديل `Candle`** | قد يكسر التحقق بصمت | `test_invalid_low` يفشل فوراً |
| **تعديل `SQLiteStorage`** | قد يفقد الدقة | `test_save_and_get` يفشل فوراً |
| **تعديل `ForexService`** | قد ينسى `limit=1` | `assert_called_once_with` يفشل فوراً |
| **إضافة حقل لـ `Candle`** | `to_dict` قد يكسر | `test_from_dict_to_dict` يفشل |
| **التأكد من الرفض** | لا شيء | `assertRaises` |
| **عزل الاختبار** | يعتمد على DB حقيقي | `:memory:` |
| **سرعة الاختبار** | دقائق | أجزاء من الثانية |

---

## 🗺️ مكانه في خريطة Sprint 1

```mermaid
graph LR
    A[core/candle.py<br/>✅] --> B[api/client.py<br/>✅]
    B --> C[api/sources/twelvedata.py<br/>✅]
    C --> D[storage/sqlite_storage.py<br/>✅]
    D --> E[services/forex_service.py<br/>✅]
    E --> F[cli/commands.py + parser.py<br/>✅]
    F --> G[tests/test_price_mvp.py<br/>✅]
    G --> H[🎉 Sprint 1 مكتمل]

    style G fill:#bfb,stroke:#333,stroke-width:3px
    style H fill:#ff9,stroke:#333,stroke-width:3px
```

---

## 💡 تشغيل الاختبارات

```bash
# كل الاختبارات
python -m unittest tests/test_price_mvp.py

# مع تفاصيل
python -m unittest -v tests/test_price_mvp.py

# فئة محددة
python -m unittest tests.test_price_mvp.TestCandleModel -v

# اختبار واحد
python -m unittest tests.test_price_mvp.TestCandleModel.test_candle_creation_valid -v
```

**المتوقع:**
```
......
----------------------------------------------------------------------
Ran 6 tests in 0.0xxs

OK
```

---

## 🧭 الفلسفة المعمارية في جملة

> **"`test_price_mvp.py` لا يختبر 'هل الكود يعمل؟' — بل يختبر 'هل الكود سيبقى يعمل عندما نغيّره غداً؟'. كل `assert` هنا هو **عقد** مع المستقبل: هذا السلوك يجب ألا ينكسر أبداً."**